# FIAP – Fase 6 | Capítulo 1 – FarmTech Solutions
## Visão Computacional com YOLOv5 (Detecção de Objetos)
**Autor:** Deivisson Gonçalves Lima – **RM565095**  
**Grupo:** 47 (Individual)  
**Notebook:** `DeivissonLima_RM565095_fase6_cap1.ipynb`  
**Período:** 10/09/2025 a 14/10/2025

### Classes escolhidas
- `copo` (Coffee cup)
- `controle` (Remote control)


# Fase 6 • Cap 1 — **Pipeline corrigido e comentado** (Open Images → YOLOv5)

Este notebook executa a **preparação completa do dataset**, com comentários em **cada etapa**, para que você possa explicar no vídeo e no relatório:

1. **Configuração** (opcional): desabilitar W&B e corrigir versões do albumentations/pydantic (evita erros no Colab CPU)  
2. **Imports e caminhos**  
3. **Carregar Open Images V7** (apenas as *classes do projeto*)  
4. **Descobrir automaticamente** o campo de detecções (`det_field`, ex.: `ground_truth`)  
5. **Filtrar** pelas classes e checar se há *bboxes*  
6. **Amostragem** balanceada por classe (40/40)  
7. **Splits** 32/4/4 por classe → total 64/8/8  
8. **Views** por split (train/val/test)  
9. **Exportar** no formato **YOLOv5** (com **cópia** de mídias)  
10. **Flatten** de subpastas criadas pelo exporter (defensivo)  
11. **Conferência** de contagens por split  
12. **Gerar `data.yaml`** (usado pelo YOLOv5)  
13. **Próximos passos**: comandos de **treino/validação/inferência** (CPU)  

## 1) (Opcional) Ambiente limpo para Colab/CPU

In [ ]:
# ✅ Dica: deixar o runtime do Colab estável para CPU
# - Desabilita o prompt do Weights & Biases (W&B)
# - Corrige versões do albumentations/pydantic para evitar erro de validação

%env WANDB_DISABLED=true
%pip -q install "albumentations==1.3.1" "pydantic<2"


## 2) Imports e caminhos-base

In [ ]:
# Imports necessários para este notebook
import os, glob, random, shutil, yaml
import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F

# Caminhos no runtime do Colab para salvar dataset exportado em formato YOLOv5
DATA_DIR_LOCAL = "/content/fase6_data"  # pasta raiz dos splits
os.makedirs(DATA_DIR_LOCAL, exist_ok=True)

# Classes do projeto (nomes exatamente como no Open Images)
CLASSES = ["Coffee cup", "Remote control"]
print("DATA_DIR_LOCAL:", DATA_DIR_LOCAL)
print("CLASSES:", CLASSES)


## 3) Carregar Open Images V7 com FiftyOne

In [ ]:
# Carrega o split 'train' do Open Images V7 contendo apenas as classes definidas.
# Usamos 'only_matching=True' para baixar SOMENTE imagens que tenham pelo menos uma bbox das classes escolhidas.
# O cache do FiftyOne acelera execuções repetidas.
ds = foz.load_zoo_dataset(
    "open-images-v7",
    split="train",
    label_types=["detections"],
    classes=CLASSES,
    only_matching=True,
    max_samples=3000,   # teto para limitar o download
    shuffle=True,
    seed=51,
)
print(ds)


## 4) Detectar automaticamente o campo de detecções (`det_field`)

In [ ]:
# O nome do campo de detecções varia por dataset (ex.: 'ground_truth' no Open Images).
# Esta rotina percorre o schema e encontra o campo cujo tipo é 'Detections'.
schema = ds.get_field_schema()
det_field = None
for fname, ftype in schema.items():
    if "Detections" in str(ftype):
        det_field = fname
        break
assert det_field, f"Nenhum campo de Detections encontrado. Schema: {schema}"
print("Campo de detecções detectado:", det_field)


## 5) Filtrar por classes e checar bboxes (>0)

In [ ]:
# Mantemos somente labels das classes-alvo neste campo
base = ds.filter_labels(det_field, F("label").is_in(CLASSES))

# Quantas amostras têm pelo menos uma detecção?
nonempty = base.match(F(f"{det_field}.detections").length() > 0)
print("Amostras com ao menos 1 bbox:", len(nonempty))


## 6) Amostrar 40 imagens por classe (garantindo bbox)

In [ ]:
def pick_ids_for_class(view, class_name, k, exclude_ids=set(), seed=7):
    """Seleciona até k IDs contendo pelo menos 1 bbox da classe informada."""
    v = (view.filter_labels(det_field, F("label") == class_name)
            .match(F(f"{det_field}.detections").length() > 0)
            .shuffle(seed=seed))
    ids = []
    for _id in v.values("id"):
        if _id not in exclude_ids:
            ids.append(_id)
            if len(ids) >= k:
                break
    return ids

ids_A = pick_ids_for_class(base, "Coffee cup", 40, set(), seed=11)
ids_B = pick_ids_for_class(base, "Remote control", 40, set(ids_A), seed=13)
print("Selecionadas -> Coffee cup:", len(ids_A), "| Remote control:", len(ids_B))
assert len(ids_A) >= 40 and len(ids_B) >= 40, "Não foi possível obter 40 amostras por classe."


## 7) Gerar splits 32/4/4 por classe (train/val/test)

In [ ]:
def split_32_4_4(ids, seed=99):
    """Embaralha e retorna: 32 (train), 4 (val), 4 (test)."""
    r = ids[:]
    random.Random(seed).shuffle(r)
    return r[:32], r[32:36], r[36:40]

train_A, val_A, test_A = split_32_4_4(ids_A)
train_B, val_B, test_B = split_32_4_4(ids_B)

train_ids = train_A + train_B
val_ids   = val_A   + val_B
test_ids  = test_A  + test_B

print("Tamanhos -> train:", len(train_ids), "| val:", len(val_ids), "| test:", len(test_ids))


## 8) Construir as views finais por split

In [ ]:
train_view = base.select(train_ids).filter_labels(det_field, F("label").is_in(CLASSES))
val_view   = base.select(val_ids).filter_labels(det_field, F("label").is_in(CLASSES))
test_view  = base.select(test_ids).filter_labels(det_field, F("label").is_in(CLASSES))

print("Views prontas:")
print("  train_view:", len(train_view))
print("  val_view:", len(val_view))
print("  test_view:", len(test_view))


## 9) Exportar cada split no formato YOLOv5 (copiando mídias)

In [ ]:
# Limpa a pasta de export e recria
shutil.rmtree(DATA_DIR_LOCAL, ignore_errors=True)
os.makedirs(DATA_DIR_LOCAL, exist_ok=True)

def export_split(view, split):
    """Exporta um split para o formato YOLOv5, copiando as imagens/labels."""
    outdir = os.path.join(DATA_DIR_LOCAL, split)
    os.makedirs(os.path.join(outdir, "images"), exist_ok=True)
    os.makedirs(os.path.join(outdir, "labels"), exist_ok=True)

    view.export(
        export_dir=outdir,
        dataset_type=fo.types.YOLOv5Dataset,
        label_field=det_field,
        classes=CLASSES,
        export_media=True,   # copia os arquivos
        overwrite=True,
    )

export_split(train_view, "train")
export_split(val_view, "val")
export_split(test_view, "test")
print("Export concluído em:", DATA_DIR_LOCAL)


## 10) **Flatten**: mover arquivos se o exporter criou subpastas `images/<split>` e `labels/<split>`

In [ ]:
def flatten_if_needed(split):
    outdir = os.path.join(DATA_DIR_LOCAL, split)
    for sub in ["images", "labels"]:
        deep = os.path.join(outdir, sub, split)
        if os.path.isdir(deep):
            for fn in os.listdir(deep):
                os.replace(os.path.join(deep, fn), os.path.join(outdir, sub, fn))
            shutil.rmtree(deep, ignore_errors=True)

for s in ["train","val","test"]:
    flatten_if_needed(s)
print("Flatten (se necessário) aplicado.")


## 11) Conferência: contagem por split

In [ ]:
for s in ["train","val","test"]:
    n_img = len(glob.glob(f"{DATA_DIR_LOCAL}/{s}/images/*"))
    n_lbl = len(glob.glob(f"{DATA_DIR_LOCAL}/{s}/labels/*"))
    print(f"{s}: imgs={n_img} | labels={n_lbl}")


## 12) Gerar `data.yaml` para o YOLOv5

In [ ]:
data_yaml = {
    "train": f"{DATA_DIR_LOCAL}/train/images",
    "val":   f"{DATA_DIR_LOCAL}/val/images",
    "test":  f"{DATA_DIR_LOCAL}/test/images",
    "nc": 2,
    "names": ["copo", "controle"],
}
with open(f"{DATA_DIR_LOCAL}/data.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False, allow_unicode=True)

print("data.yaml salvo em:", f"{DATA_DIR_LOCAL}/data.yaml")


## 13) Próximos passos — Treino/Validação/Inferência (CPU)

In [ ]:
# Execute num notebook onde o YOLOv5 já foi clonado (%cd /content/yolov5) e deps instaladas.
# Treino rápido (30 épocas) — CPU
%cd /content/yolov5
!python train.py   --img 448 --batch 8 --epochs 30   --data /content/fase6_data/data.yaml   --weights yolov5n.pt   --cache ram --workers 2 --device cpu   --freeze 10 --patience 10   --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs   --name exp_e30_fast_cpu --exist-ok

# Validação e inferência de exemplo (CPU)
!python val.py --weights /content/drive/MyDrive/Fase6/Fase6_Cap1/runs/exp_e30_fast_cpu/weights/best.pt --data /content/fase6_data/data.yaml --task val --device cpu --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs --name exp_e30_val --exist-ok
!python detect.py --weights /content/drive/MyDrive/Fase6/Fase6_Cap1/runs/exp_e30_fast_cpu/weights/best.pt --img 448 --conf 0.25 --source /content/fase6_data/val/images --device cpu --project /content/drive/MyDrive/Fase6/Fase6_Cap1/runs --name infer_val_e30 --exist-ok
